In [ ]:
import math
import itertools
import pickle
import random
import numpy as np
import torch
import tqdm
import enum
from pathlib import Path
from dataclasses import dataclass
from PIL import Image
import matplotlib.pyplot as plt

class MoonInfoOrigin(enum.Enum):
    DIRECT = 0
    INTERPOLATED = 1


@dataclass
class ImageInfo:
    path: Path
    width: int
    height: int
    avg_brightness: float
    timestamp: float
    exposure_time: float
    moon: tuple[float, float, float] = None
    moon_info_origin: MoonInfoOrigin = None
    moon_pos_std_px: float = None

# Load registration data (from eda00.py output)
with open("/home/slavik/tmp/eda00.pkl", "rb") as fd:
    exposure_groups = pickle.load(fd)
    reg = pickle.load(fd)

In [ ]:
def run_group(exposure_time, n, reg_exp, device, n_iter=100_000, peak_lr=1e-3, warmup_frac=0.1):
    """
    reg_exp: dict (i, j) -> (shift_i, shift_j, rotation_deg) for transform from j to i.
    Returns (loss_shift_final, loss_rot_final, abs_xy, abs_angle).
    """
    # Greedy init: find pair with smallest shift magnitude
    best_ij = None
    best_mag = float("inf")
    for (i, j) in reg_exp:
        s_i, s_j, _ = reg_exp[(i, j)]
        mag = math.sqrt(s_i**2 + s_j**2)
        if mag < best_mag:
            best_mag = mag
            best_ij = (i, j)
    i0, j0 = best_ij
    s_i, s_j, r = reg_exp[(i0, j0)]
    # i0 at (0, 0), angle 0. j0: implied_rot = theta_i0 - theta_j0 = r => theta_j0 = -r.
    # implied_shift = R(-theta_i0)((x_j0, y_j0)) = (x_j0, y_j0) = (s_i, s_j)
    placed = {i0, j0}
    abs_x = [0.0] * n
    abs_y = [0.0] * n
    abs_angle = [0.0] * n
    abs_x[i0], abs_y[i0], abs_angle[i0] = 0.0, 0.0, 0.0
    abs_x[j0], abs_y[j0], abs_angle[j0] = s_i, s_j, math.radians(-r)

    while len(placed) < n:
        best_k = None
        best_ref = None
        best_mag = float("inf")
        for k in range(n):
            if k in placed:
                continue
            for ref in placed:
                if (ref, k) not in reg_exp:
                    continue
                s_i, s_j, _ = reg_exp[(ref, k)]
                mag = math.sqrt(s_i**2 + s_j**2)
                if mag < best_mag:
                    best_mag = mag
                    best_k = k
                    best_ref = ref
        if best_k is None:
            break
        ref, k = best_ref, best_k
        s_i, s_j, r_deg = reg_exp[(ref, k)]
        r_rad = math.radians(r_deg)
        # implied_shift = R(-theta_ref)((x_k - x_ref, y_k - y_ref)) = (s_i, s_j) => (x_k - x_ref, y_k - y_ref) = R(theta_ref)(s_i, s_j)
        # implied_rot = theta_ref - theta_k = r_rad => theta_k = theta_ref - r_rad
        theta_ref = abs_angle[ref]
        dx = math.cos(theta_ref) * s_i - math.sin(theta_ref) * s_j
        dy = math.sin(theta_ref) * s_i + math.cos(theta_ref) * s_j
        theta_k = theta_ref - r_rad
        abs_x[k] = abs_x[ref] + dx
        abs_y[k] = abs_y[ref] + dy
        abs_angle[k] = theta_k
        placed.add(k)

    # Torch params
    abs_xy = torch.tensor([[abs_x[i], abs_y[i]] for i in range(n)], dtype=torch.float32, device=device, requires_grad=True)
    abs_angle_t = torch.tensor([abs_angle[i] for i in range(n)], dtype=torch.float32, device=device, requires_grad=True)

    pairs = list(reg_exp.keys())
    reg_shift_i = torch.tensor([reg_exp[(i, j)][0] for (i, j) in pairs], dtype=torch.float32, device=device)
    reg_shift_j = torch.tensor([reg_exp[(i, j)][1] for (i, j) in pairs], dtype=torch.float32, device=device)
    reg_rot = torch.tensor([math.radians(reg_exp[(i, j)][2]) for (i, j) in pairs], dtype=torch.float32, device=device)
    idx_i = torch.tensor([i for (i, j) in pairs], dtype=torch.long, device=device)
    idx_j = torch.tensor([j for (i, j) in pairs], dtype=torch.long, device=device)

    def loss_fn():
        # Implied from j to i: shift_ij = R(-theta_i)((x_j - x_i, y_j - y_i)) (j's center in i's frame), rot_ij = theta_i - theta_j
        x_i = abs_xy[idx_i, 0]
        y_i = abs_xy[idx_i, 1]
        x_j = abs_xy[idx_j, 0]
        y_j = abs_xy[idx_j, 1]
        theta_i = abs_angle_t[idx_i]
        theta_j = abs_angle_t[idx_j]
        dx = x_j - x_i
        dy = y_j - y_i
        ci = torch.cos(-theta_i)
        si = torch.sin(-theta_i)
        impl_shift_i = ci * dx - si * dy
        impl_shift_j = si * dx + ci * dy
        impl_rot = theta_i - theta_j
        loss_shift = ((impl_shift_i - reg_shift_i) ** 2 + (impl_shift_j - reg_shift_j) ** 2).sum()
        loss_rot = ((impl_rot - reg_rot) ** 2).sum()
        return loss_shift, loss_rot

    def lr_schedule(step, n_steps):
        if warmup_frac > 0 and step < n_steps * warmup_frac:
            return peak_lr * (step / (n_steps * warmup_frac))
        progress = (step - n_steps * warmup_frac) / max(1, n_steps * (1 - warmup_frac))
        return 0.5 * peak_lr * (1 + math.cos(math.pi * min(1.0, progress)))

    n_phase = 50_000
    with torch.no_grad():
        ls0, lr0 = loss_fn()
    print(f"  exp={exposure_time}: initial loss_shift={ls0.item():.6f} loss_rot={lr0.item():.6f}")

    # Phase 1: optimize angles only (rotation loss), 50k iters
    opt_rot = torch.optim.Adam([abs_angle_t], lr=peak_lr)
    for step in tqdm.tqdm(range(n_phase), desc="angles"):
        opt_rot.zero_grad()
        for g in opt_rot.param_groups:
            g["lr"] = lr_schedule(step, n_phase)
        _, loss_rot = loss_fn()
        loss_rot.backward()
        opt_rot.step()

    with torch.no_grad():
        ls0, lr0 = loss_fn()
    print(f"  exp={exposure_time}: phase1 loss_shift={ls0.item():.6f} loss_rot={lr0.item():.6f}")

    # Phase 2: optimize positions only (shift loss), 50k iters
    abs_angle_t.requires_grad_(False)
    opt_shift = torch.optim.Adam([abs_xy], lr=peak_lr)
    for step in tqdm.tqdm(range(n_phase), desc="shifts"):
        opt_shift.zero_grad()
        for g in opt_shift.param_groups:
            g["lr"] = lr_schedule(step, n_phase)
        loss_shift, _ = loss_fn()
        loss_shift.backward()
        opt_shift.step()

    with torch.no_grad():
        ls1, lr1 = loss_fn()
    print(f"  exp={exposure_time}: final   loss_shift={ls1.item():.6f} loss_rot={lr1.item():.6f}")
    return ls1.item(), lr1.item(), abs_xy.detach(), abs_angle_t.detach()

In [ ]:
def load_grayscale(ii, device):
    with Image.open(ii.path) as img:
        arr = np.array(img).astype(np.float32) / 255.0
    if arr.ndim == 3:
        arr = arr.mean(axis=2)
    return torch.from_numpy(arr).to(device=device, dtype=torch.float32)

def apply_transform_single(img, shift_i, shift_j, angle_deg, device):
    H, W = img.shape
    ci, cj = H / 2.0, W / 2.0
    angle_rad = math.radians(-angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
    ii = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1).expand(H, W)
    jj = torch.arange(W, device=device, dtype=torch.float32).view(1, -1).expand(H, W)
    di = ii - ci - shift_i
    dj = jj - cj - shift_j
    i_src = di * cos_a + dj * sin_a + ci
    j_src = -di * sin_a + dj * cos_a + cj
    j_norm = 2.0 * j_src / (W - 1) - 1.0 if W > 1 else torch.zeros_like(j_src)
    i_norm = 2.0 * i_src / (H - 1) - 1.0 if H > 1 else torch.zeros_like(i_src)
    grid = torch.stack([j_norm, i_norm], dim=-1).unsqueeze(0)
    out = torch.nn.functional.grid_sample(
        img.unsqueeze(0).unsqueeze(0), grid, mode="bilinear", padding_mode="zeros", align_corners=True
    )
    return out.squeeze(0).squeeze(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_iter = 100_000
peak_lr = 1e-3
warmup_frac = 0.1

for exposure_time in sorted(exposure_groups.keys()):
    group = list(exposure_groups[exposure_time])
    n = len(group)
    if n < 2:
        continue
    reg_exp = {(i, j): reg[(exposure_time, i, j)] for (i, j) in itertools.permutations(range(n), 2) if (exposure_time, i, j) in reg}
    if not reg_exp:
        continue
    _, _, abs_xy, abs_angle_t = run_group(exposure_time, n, reg_exp, device, n_iter=n_iter, peak_lr=peak_lr, warmup_frac=warmup_frac)

    # Debug plot: single random image | average of all registered (crop 2.4*moon_radius, moon from random image)
    idx_show = random.randint(0, n - 1)
    ii_show = group[idx_show]
    mi, mj, r = ii_show.moon[0], ii_show.moon[1], ii_show.moon[2]
    half = 1.2 * r
    H, W = ii_show.height, ii_show.width
    i_lo = max(0, int(mi - half))
    i_hi = min(H, int(mi + half))
    j_lo = max(0, int(mj - half))
    j_hi = min(W, int(mj + half))
    img_single = load_grayscale(ii_show, device)
    warped = []
    for j in range(n):
        img_j = load_grayscale(group[j], device)
        x_j = abs_xy[j, 0].item()
        y_j = abs_xy[j, 1].item()
        theta_j_deg = -math.degrees(abs_angle_t[j].item())
        w = apply_transform_single(img_j, x_j, y_j, theta_j_deg, device)
        warped.append(w)
    avg_img = torch.stack(warped, dim=0).mean(dim=0)
    crop_single = img_single[i_lo:i_hi, j_lo:j_hi].cpu().numpy()
    crop_avg = avg_img[i_lo:i_hi, j_lo:j_hi].cpu().numpy()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    ax1.imshow(crop_single, cmap="gray", vmin=0, vmax=1)
    ax1.set_title(f"exp={exposure_time}: single (idx {idx_show})")
    ax1.axis("off")
    ax2.imshow(crop_avg, cmap="gray", vmin=0, vmax=1)
    ax2.set_title(f"exp={exposure_time}: average of {n} registered")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()
